In [1]:
import os
import re
import glob

from pydub import AudioSegment
from pydub.silence import detect_silence

import torch
import torchaudio

/rds/general/user/ak8224/home/miniforge3/envs/emojivoice/lib/python3.10/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


## Use Filtered Scripts with TTS Ready Format

In [2]:
def clean_aphasia_transcript_for_tts(text):
    """
    Translates AphasiaBank CHAT formatting into TTS-compatible custom tokens.
    Maps clinical prosody markers to specific punctuation (@, #, etc.)
    """
    # 1. Strip out the participant label and timestamps
    text = text.replace("*PAR:", "")
    text = re.sub(r'&\*[A-Z]+:', '', text)
    
    # 2. THE ABANDONMENT TOKEN (#)
    # Maps trailing off (+...) or interrupted (+/?) to a hashtag
    text = re.sub(r'\+\.\.\.|\+\/\?', '#', text)
    
    # 3. THE STUTTER/RESTART TOKEN (@)
    # Maps retracing [/] and false starts [//] to the @ symbol
    text = re.sub(r'\[//?\]', '@', text)
    
    # 4. TRANSLATE CHAT PAUSES TO TTS PAUSES
    text = text.replace('(.)', ',')            # Short pause becomes a comma
    text = re.sub(r'\(\.\.\.?\)', '...', text) # Medium/Long pauses become ellipses
    
    # 5. KEEP FILLERS, STRIP TAGS
    # Removes &- and &+ so "&-um" cleanly becomes "um"
    text = re.sub(r'&[-+]', '', text)
    
    # 6. DELETE GESTURES AND NOISES
    # Removes tags like &=ges or &=laughs, leaving the surrounding words intact
    text = re.sub(r'&=\S+', '', text)
    
    # 7. PURGE REMAINING TRANSCRIBER NOTES
    # Now that we saved [/] and [//], we delete all other bracketed error codes (e.g., [* s])
    text = re.sub(r'\[.*?\]', '', text)
    
    # 8. REMOVE SCOPE BRACKETS
    # Deletes < and > so "<I didn't> @" cleanly becomes "I didn't @"
    text = re.sub(r'[<>]', '', text)
    
    # 9. FINAL TTS TEXT CLEANUP
    # Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text)
    # Snap floating punctuation to the preceding word (e.g., "work , " -> "work, ")
    text = re.sub(r'\s+([,#\.?!])', r'\1', text)
    
    return text.strip()

In [3]:
example = "&-um &-uh I'm [/] &+i I'm <going to> [//] &-uh in Dothan [/] &-um &*INV:Dothan Alabama ."
print(f"\n🔍 Test Original: {example}")
print(f"✨ Test Cleaned:  {clean_aphasia_transcript_for_tts(example)}")


🔍 Test Original: &-um &-uh I'm [/] &+i I'm <going to> [//] &-uh in Dothan [/] &-um &*INV:Dothan Alabama .
✨ Test Cleaned:  um uh I'm @ i I'm going to @ uh in Dothan @ um Dothan Alabama.


In [4]:
# --- BATCH PROCESS DIRECTORY ---
dataset_dir = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts"
txt_files = glob.glob(f"{dataset_dir}/*.txt")

processed_dir = os.path.join(dataset_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)

print(f"🧹 Found {len(txt_files)} transcripts to clean. Starting...")

for file_path in txt_files:
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    cleaned_text = clean_aphasia_transcript_for_tts(raw_text)
    filename = os.path.basename(file_path)
    new_save_path = os.path.join(processed_dir, filename)
    
    with open(new_save_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

print("="*60)
print(f"✅ All transcripts successfully sanitized and safely stored in:\n{processed_dir}")
print("="*60)

🧹 Found 77 transcripts to clean. Starting...
✅ All transcripts successfully sanitized and safely stored in:
/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts/processed


In [9]:
# --- BATCH PROCESS DIRECTORY ---
dataset_dir = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts2"
txt_files = glob.glob(f"{dataset_dir}/*.txt")

processed_dir = os.path.join(dataset_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)

print(f"🧹 Found {len(txt_files)} transcripts to clean. Starting...")

for file_path in txt_files:
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    cleaned_text = clean_aphasia_transcript_for_tts(raw_text)
    filename = os.path.basename(file_path)
    new_save_path = os.path.join(processed_dir, filename)
    
    with open(new_save_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

print("="*60)
print(f"✅ All transcripts successfully sanitized and safely stored in:\n{processed_dir}")
print("="*60)

🧹 Found 73 transcripts to clean. Starting...
✅ All transcripts successfully sanitized and safely stored in:
/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts2/processed


In [14]:
# --- BATCH PROCESS DIRECTORY ---
dataset_dir = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts"
txt_files = glob.glob(f"{dataset_dir}/*.txt")

processed_dir = os.path.join(dataset_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)

print(f"🧹 Found {len(txt_files)} transcripts to clean. Starting...")

for file_path in txt_files:
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    cleaned_text = clean_aphasia_transcript_for_tts(raw_text)
    filename = os.path.basename(file_path)
    new_save_path = os.path.join(processed_dir, filename)
    
    with open(new_save_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

print("="*60)
print(f"✅ All transcripts successfully sanitized and safely stored in:\n{processed_dir}")
print("="*60)

🧹 Found 60 transcripts to clean. Starting...
✅ All transcripts successfully sanitized and safely stored in:
/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts/processed


In [18]:
# --- BATCH PROCESS DIRECTORY ---
dataset_dir = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts2"
txt_files = glob.glob(f"{dataset_dir}/*.txt")

processed_dir = os.path.join(dataset_dir, "processed")
os.makedirs(processed_dir, exist_ok=True)

print(f"🧹 Found {len(txt_files)} transcripts to clean. Starting...")

for file_path in txt_files:
    with open(file_path, "r", encoding="utf-8") as f:
        raw_text = f.read()
        
    cleaned_text = clean_aphasia_transcript_for_tts(raw_text)
    filename = os.path.basename(file_path)
    new_save_path = os.path.join(processed_dir, filename)
    
    with open(new_save_path, "w", encoding="utf-8") as f:
        f.write(cleaned_text)

print("="*60)
print(f"✅ All transcripts successfully sanitized and safely stored in:\n{processed_dir}")
print("="*60)

🧹 Found 46 transcripts to clean. Starting...
✅ All transcripts successfully sanitized and safely stored in:
/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts2/processed


## Preprocessed Audio

In [5]:
def preprocess_aphasia_audio_preserve_silences(input_dir):
    """
    Normalizes volume to -23 dBFS, converts to 22050Hz Mono, 
    and scans for long silences without cutting them.
    """
    output_dir = os.path.join(input_dir, "preprocessed")
    os.makedirs(output_dir, exist_ok=True)

    wav_files = glob.glob(os.path.join(input_dir, "*.wav"))
    print(f"🎧 Found {len(wav_files)} files. Starting normalization and silence scanning...\n")

    pause_report = {}

    for file_path in wav_files:
        filename = os.path.basename(file_path)
        out_path = os.path.join(output_dir, filename)

        # --- 1. LOAD AUDIO ---
        audio = AudioSegment.from_wav(file_path)

        # --- 2. VOLUME NORMALIZATION ---
        target_dBFS = -23.0
        gain_change = target_dBFS - audio.dBFS
        normalized_audio = audio.apply_gain(gain_change)

        # --- 3. SILENCE DETECTION ---
        silences = detect_silence(
            normalized_audio,
            min_silence_len=1000, 
            silence_thresh=-40
        )

        # --- 4. LOG THE PAUSES ---
        if silences:
            pause_report[filename] = []
            for start_ms, end_ms in silences:
                duration_sec = (end_ms - start_ms) / 1000.0
                start_sec = start_ms / 1000.0
                pause_report[filename].append((start_sec, duration_sec))

        # --- 5. EXPORT FOR TTS ---
        normalized_audio.export(out_path, format="wav", parameters=["-ar", "22050", "-ac", "1"])

    # --- 6. PRINT THE MASTER PAUSE REPORT ---
    print("\n" + "="*60)
    print("🛑 LONG PAUSE DETECTION REPORT")
    print("Use this to manually verify your '...' token placements!")
    print("="*60)
    
    if not pause_report:
        print("No long pauses (>1s) found in any files.")
    else:
        for fname, pauses in pause_report.items():
            print(f"\n📄 {fname}")
            for start_sec, duration_sec in pauses:
                print(f"   - Pause starts at {start_sec:.2f}s | Duration: {duration_sec:.2f} seconds")
                
    print("="*60)
    print(f"🎉 All preprocessed audio safely saved to:\n{output_dir}")

In [15]:
TARGET_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected"
preprocess_aphasia_audio_preserve_silences(TARGET_DIR)

🎧 Found 77 files. Starting normalization and silence scanning...


🛑 LONG PAUSE DETECTION REPORT
Use this to manually verify your '...' token placements!

📄 aphasia5_0208.wav
   - Pause starts at 8.68s | Duration: 1.93 seconds
   - Pause starts at 12.29s | Duration: 1.89 seconds
   - Pause starts at 15.68s | Duration: 1.74 seconds

📄 aphasia5_0063.wav
   - Pause starts at 6.14s | Duration: 1.25 seconds
   - Pause starts at 9.84s | Duration: 1.40 seconds

📄 aphasia5_0129.wav
   - Pause starts at 0.17s | Duration: 1.15 seconds
   - Pause starts at 5.00s | Duration: 1.01 seconds

📄 aphasia5_0170.wav
   - Pause starts at 1.54s | Duration: 1.01 seconds
   - Pause starts at 2.83s | Duration: 1.79 seconds
   - Pause starts at 6.15s | Duration: 1.22 seconds
   - Pause starts at 7.68s | Duration: 2.44 seconds
   - Pause starts at 11.53s | Duration: 2.13 seconds
   - Pause starts at 14.51s | Duration: 2.13 seconds
   - Pause starts at 17.85s | Duration: 1.59 seconds
   - Pause starts at 22.45s |

In [6]:
TARGET_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected2"
preprocess_aphasia_audio_preserve_silences(TARGET_DIR)

🎧 Found 73 files. Starting normalization and silence scanning...


🛑 LONG PAUSE DETECTION REPORT
Use this to manually verify your '...' token placements!

📄 aphasia5_0071.wav
   - Pause starts at 2.22s | Duration: 1.34 seconds

📄 aphasia5_0185.wav
   - Pause starts at 3.39s | Duration: 1.19 seconds

📄 aphasia5_0196.wav
   - Pause starts at 0.46s | Duration: 2.16 seconds

📄 aphasia5_0008.wav
   - Pause starts at 2.18s | Duration: 1.00 seconds

📄 aphasia5_0054.wav
   - Pause starts at 0.99s | Duration: 1.16 seconds

📄 aphasia5_0161.wav
   - Pause starts at 0.92s | Duration: 1.33 seconds

📄 aphasia5_0158.wav
   - Pause starts at 0.27s | Duration: 1.20 seconds
🎉 All preprocessed audio safely saved to:
/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected2/preprocessed


In [15]:
TARGET_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected"
preprocess_aphasia_audio_preserve_silences(TARGET_DIR)

🎧 Found 60 files. Starting normalization and silence scanning...


🛑 LONG PAUSE DETECTION REPORT
Use this to manually verify your '...' token placements!

📄 aphasia3_0143.wav
   - Pause starts at 0.43s | Duration: 1.36 seconds
   - Pause starts at 2.31s | Duration: 1.84 seconds

📄 aphasia3_0059.wav
   - Pause starts at 0.38s | Duration: 1.08 seconds
   - Pause starts at 2.83s | Duration: 1.01 seconds

📄 aphasia3_0050.wav
   - Pause starts at 0.46s | Duration: 1.59 seconds

📄 aphasia3_0155.wav
   - Pause starts at 1.62s | Duration: 1.62 seconds

📄 aphasia3_0142.wav
   - Pause starts at 0.93s | Duration: 3.17 seconds
   - Pause starts at 12.38s | Duration: 1.27 seconds
   - Pause starts at 19.61s | Duration: 2.09 seconds

📄 aphasia3_0025.wav
   - Pause starts at 1.66s | Duration: 1.09 seconds

📄 aphasia3_0052.wav
   - Pause starts at 3.11s | Duration: 1.15 seconds

📄 aphasia3_0126.wav
   - Pause starts at 2.25s | Duration: 2.91 seconds
   - Pause starts at 7.68s | Duration: 1.02 seconds


In [19]:
TARGET_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected2"
preprocess_aphasia_audio_preserve_silences(TARGET_DIR)

🎧 Found 46 files. Starting normalization and silence scanning...


🛑 LONG PAUSE DETECTION REPORT
Use this to manually verify your '...' token placements!

📄 aphasia3_0011.wav
   - Pause starts at 0.47s | Duration: 1.09 seconds
   - Pause starts at 2.90s | Duration: 1.25 seconds

📄 aphasia3_0088.wav
   - Pause starts at 1.15s | Duration: 1.72 seconds
   - Pause starts at 5.89s | Duration: 1.07 seconds

📄 aphasia3_0094.wav
   - Pause starts at 2.65s | Duration: 1.71 seconds
   - Pause starts at 4.58s | Duration: 1.42 seconds

📄 aphasia3_0131.wav
   - Pause starts at 3.56s | Duration: 1.77 seconds

📄 aphasia3_0115.wav
   - Pause starts at 0.49s | Duration: 1.99 seconds

📄 aphasia3_0100.wav
   - Pause starts at 0.70s | Duration: 3.03 seconds
   - Pause starts at 5.02s | Duration: 1.03 seconds

📄 aphasia3_0103.wav
   - Pause starts at 1.19s | Duration: 1.42 seconds
🎉 All preprocessed audio safely saved to:
/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/selected2/prep

## Create Final Metadata

In [16]:
# --- 1. SETUP PATHS ---
hpc_root = "/rds/general/user/ak8224/home"

# Directory containing your final 100 preprocessed audio files
wavs_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia5/selected/preprocessed"

# Directory containing your cleaned text scripts
scripts_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts/processed"

# Where to save the final metadata file
output_metadata = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia5/metadata.csv"

# --- 2. GATHER AUDIO FILES ---
wav_files = glob.glob(os.path.join(wavs_dir, "*.wav"))
print(f"🔍 Found {len(wav_files)} audio files in the preprocessed directory.")

metadata_lines = []
missing_scripts = 0

# --- 3. MATCH AND BUILD ---
for wav_path in wav_files:
    # Extract just the filename without the .wav extension (e.g., "patient_0001")
    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    
    # Build the path to where the matching text file *should* be
    txt_path = os.path.join(scripts_dir, f"{base_name}.txt")
    
    # Check if the text file actually exists in the script directory
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            # Read the transcript and strip any accidental whitespace/newlines
            transcript = f.read().strip()
            
        # Format: <absolute_wav_path>|<speaker_id>|<transcript>
        line = f"{wav_path}|0|{transcript}"
        metadata_lines.append(line)
    else:
        print(f"⚠️ Warning: Could not find matching transcript for {base_name}.wav")
        missing_scripts += 1

# --- 4. SAVE TO CSV ---
with open(output_metadata, "w", encoding="utf-8") as f:
    for line in metadata_lines:
        f.write(line + "\n")

# --- 5. REPORTING ---
print(f"\n✅ Successfully generated metadata.csv with {len(metadata_lines)} entries!")
if missing_scripts > 0:
    print(f"ℹ️ Skipped {missing_scripts} audio files because their text files were missing.")
print(f"📄 Saved to: {output_metadata}")

🔍 Found 77 audio files in the preprocessed directory.

✅ Successfully generated metadata.csv with 77 entries!
📄 Saved to: /rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/metadata.csv


In [7]:
# --- 1. SETUP PATHS ---
hpc_root = "/rds/general/user/ak8224/home"

# Directory containing your final 100 preprocessed audio files
wavs_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia5/selected2/preprocessed"

# Directory containing your cleaned text scripts
scripts_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia5/filtered_scripts2/processed"

# Where to save the final metadata file
output_metadata = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia5/metadata2.csv"

# --- 2. GATHER AUDIO FILES ---
wav_files = glob.glob(os.path.join(wavs_dir, "*.wav"))
print(f"🔍 Found {len(wav_files)} audio files in the preprocessed directory.")

metadata_lines = []
missing_scripts = 0

# --- 3. MATCH AND BUILD ---
for wav_path in wav_files:
    # Extract just the filename without the .wav extension (e.g., "patient_0001")
    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    
    # Build the path to where the matching text file *should* be
    txt_path = os.path.join(scripts_dir, f"{base_name}.txt")
    
    # Check if the text file actually exists in the script directory
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            # Read the transcript and strip any accidental whitespace/newlines
            transcript = f.read().strip()
            
        # Format: <absolute_wav_path>|<speaker_id>|<transcript>
        line = f"{wav_path}|0|{transcript}"
        metadata_lines.append(line)
    else:
        print(f"⚠️ Warning: Could not find matching transcript for {base_name}.wav")
        missing_scripts += 1

# --- 4. SAVE TO CSV ---
with open(output_metadata, "w", encoding="utf-8") as f:
    for line in metadata_lines:
        f.write(line + "\n")

# --- 5. REPORTING ---
print(f"\n✅ Successfully generated metadata2.csv with {len(metadata_lines)} entries!")
if missing_scripts > 0:
    print(f"ℹ️ Skipped {missing_scripts} audio files because their text files were missing.")
print(f"📄 Saved to: {output_metadata}")

🔍 Found 73 audio files in the preprocessed directory.

✅ Successfully generated metadata2.csv with 73 entries!
📄 Saved to: /rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/metadata2.csv


In [16]:
# --- 1. SETUP PATHS ---
hpc_root = "/rds/general/user/ak8224/home"

# Directory containing your final 100 preprocessed audio files
wavs_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia3/selected/preprocessed"

# Directory containing your cleaned text scripts
scripts_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts/processed"

# Where to save the final metadata file
output_metadata = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia3/metadata.csv"

# --- 2. GATHER AUDIO FILES ---
wav_files = glob.glob(os.path.join(wavs_dir, "*.wav"))
print(f"🔍 Found {len(wav_files)} audio files in the preprocessed directory.")

metadata_lines = []
missing_scripts = 0

# --- 3. MATCH AND BUILD ---
for wav_path in wav_files:
    # Extract just the filename without the .wav extension (e.g., "patient_0001")
    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    
    # Build the path to where the matching text file *should* be
    txt_path = os.path.join(scripts_dir, f"{base_name}.txt")
    
    # Check if the text file actually exists in the script directory
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            # Read the transcript and strip any accidental whitespace/newlines
            transcript = f.read().strip()
            
        # Format: <absolute_wav_path>|<speaker_id>|<transcript>
        line = f"{wav_path}|0|{transcript}"
        metadata_lines.append(line)
    else:
        print(f"⚠️ Warning: Could not find matching transcript for {base_name}.wav")
        missing_scripts += 1

# --- 4. SAVE TO CSV ---
with open(output_metadata, "w", encoding="utf-8") as f:
    for line in metadata_lines:
        f.write(line + "\n")

# --- 5. REPORTING ---
print(f"\n✅ Successfully generated metadata.csv with {len(metadata_lines)} entries!")
if missing_scripts > 0:
    print(f"ℹ️ Skipped {missing_scripts} audio files because their text files were missing.")
print(f"📄 Saved to: {output_metadata}")

🔍 Found 60 audio files in the preprocessed directory.

✅ Successfully generated metadata.csv with 60 entries!
📄 Saved to: /rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/metadata.csv


In [20]:
# --- 1. SETUP PATHS ---
hpc_root = "/rds/general/user/ak8224/home"

# Directory containing your final 100 preprocessed audio files
wavs_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia3/selected2/preprocessed"

# Directory containing your cleaned text scripts
scripts_dir = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia3/filtered_scripts2/processed"

# Where to save the final metadata file
output_metadata = f"{hpc_root}/emoji_project/data/Aphasia-Test/aphasia3/metadata2.csv"

# --- 2. GATHER AUDIO FILES ---
wav_files = glob.glob(os.path.join(wavs_dir, "*.wav"))
print(f"🔍 Found {len(wav_files)} audio files in the preprocessed directory.")

metadata_lines = []
missing_scripts = 0

# --- 3. MATCH AND BUILD ---
for wav_path in wav_files:
    # Extract just the filename without the .wav extension (e.g., "patient_0001")
    base_name = os.path.splitext(os.path.basename(wav_path))[0]
    
    # Build the path to where the matching text file *should* be
    txt_path = os.path.join(scripts_dir, f"{base_name}.txt")
    
    # Check if the text file actually exists in the script directory
    if os.path.exists(txt_path):
        with open(txt_path, "r", encoding="utf-8") as f:
            # Read the transcript and strip any accidental whitespace/newlines
            transcript = f.read().strip()
            
        # Format: <absolute_wav_path>|<speaker_id>|<transcript>
        line = f"{wav_path}|0|{transcript}"
        metadata_lines.append(line)
    else:
        print(f"⚠️ Warning: Could not find matching transcript for {base_name}.wav")
        missing_scripts += 1

# --- 4. SAVE TO CSV ---
with open(output_metadata, "w", encoding="utf-8") as f:
    for line in metadata_lines:
        f.write(line + "\n")

# --- 5. REPORTING ---
print(f"\n✅ Successfully generated metadata2.csv with {len(metadata_lines)} entries!")
if missing_scripts > 0:
    print(f"ℹ️ Skipped {missing_scripts} audio files because their text files were missing.")
print(f"📄 Saved to: {output_metadata}")

🔍 Found 46 audio files in the preprocessed directory.

✅ Successfully generated metadata2.csv with 46 entries!
📄 Saved to: /rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia3/metadata2.csv


In [9]:
def sort_metadata_in_place(metadata_path):
    print(f"📂 Loading metadata from: {metadata_path}")
    
    with open(metadata_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        
    if not lines:
        print("❌ File is empty.")
        return

    lines.sort(key=lambda x: os.path.basename(x.split('|')[0]))

    # Overwrite the file with the perfectly sorted lines
    with open(metadata_path, 'w', encoding='utf-8') as f:
        f.writelines(lines)
        
    print(f"✅ Successfully sorted {len(lines)} lines alphabetically by filename!")

In [19]:
METADATA_FILE = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/metadata.csv"
sort_metadata_in_place(METADATA_FILE)

📂 Loading metadata from: /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/metadata.csv
✅ Successfully sorted 77 lines alphabetically by filename!


In [10]:
METADATA_FILE = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/metadata2.csv"
sort_metadata_in_place(METADATA_FILE)

📂 Loading metadata from: /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/metadata2.csv
✅ Successfully sorted 73 lines alphabetically by filename!


In [17]:
METADATA_FILE = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/metadata.csv"
sort_metadata_in_place(METADATA_FILE)

📂 Loading metadata from: /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/metadata.csv
✅ Successfully sorted 60 lines alphabetically by filename!


In [21]:
METADATA_FILE = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/metadata2.csv"
sort_metadata_in_place(METADATA_FILE)

📂 Loading metadata from: /rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/metadata2.csv
✅ Successfully sorted 46 lines alphabetically by filename!


## Create Split for Fine-Tuning

In [ ]:
import random

metadata_path = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/metadata.csv"

with open(metadata_path, 'r') as f:
    lines = [line.strip() for line in f.readlines()]

random.shuffle(lines)

split_idx = int(len(lines) * 0.95)
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/train.txt", "w") as f:
    f.write("\n".join(train_lines) + "\n")

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/val.txt", "w") as f:
    f.write("\n".join(val_lines) + "\n")

print(f"✅ Created train.txt ({len(train_lines)} files) and val.txt ({len(val_lines)} files)!")

✅ Created train.txt (73 files) and val.txt (4 files)!


In [12]:
import random

metadata_path = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/metadata2.csv"

with open(metadata_path, 'r') as f:
    lines = [line.strip() for line in f.readlines()]

random.shuffle(lines)

split_idx = int(len(lines) * 0.99)
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/train2.txt", "w") as f:
    f.write("\n".join(train_lines) + "\n")

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/val2.txt", "w") as f:
    f.write("\n".join(val_lines) + "\n")

print(f"✅ Created train2.txt ({len(train_lines)} files) and val2.txt ({len(val_lines)} files)!")

✅ Created train2.txt (72 files) and val2.txt (1 files)!


In [22]:
import random

metadata_path = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/metadata.csv"

with open(metadata_path, 'r') as f:
    lines = [line.strip() for line in f.readlines()]

random.shuffle(lines)

split_idx = int(len(lines) * 0.95)
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/train.txt", "w") as f:
    f.write("\n".join(train_lines) + "\n")

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/val.txt", "w") as f:
    f.write("\n".join(val_lines) + "\n")

print(f"✅ Created train.txt ({len(train_lines)} files) and val.txt ({len(val_lines)} files)!")

✅ Created train.txt (57 files) and val.txt (3 files)!


In [23]:
import random

metadata_path = "/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/metadata2.csv"

with open(metadata_path, 'r') as f:
    lines = [line.strip() for line in f.readlines()]

random.shuffle(lines)

split_idx = int(len(lines) * 0.99)
train_lines = lines[:split_idx]
val_lines = lines[split_idx:]

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/train2.txt", "w") as f:
    f.write("\n".join(train_lines) + "\n")

with open("/rds/general/user/ak8224/home/MedEmoji-TTS/data/aphasia_v2/male/val2.txt", "w") as f:
    f.write("\n".join(val_lines) + "\n")

print(f"✅ Created train2.txt ({len(train_lines)} files) and val2.txt ({len(val_lines)} files)!")

✅ Created train2.txt (45 files) and val2.txt (1 files)!


## Calculating Mel-Spec Statistics

In [23]:
def get_mel_stats(wav_dir):
    print(f"🔍 Scanning directory: {wav_dir}")
    wav_files = glob.glob(os.path.join(wav_dir, "*.wav"))
    
    if not wav_files:
        print("❌ No .wav files found! Check your directory path.")
        return

    print(f"📊 Found {len(wav_files)} files. Calculating Log-Mel Spectrograms...")

    # Match these EXACTLY to your aphasia.yaml data parameters
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=22050,
        n_fft=1024,
        win_length=1024,
        hop_length=256,
        f_min=0.0,
        f_max=8000.0,
        n_mels=80,
        power=1.0, 
        normalized=False
    )

    all_mels = []
    
    for f in wav_files:
        waveform, sr = torchaudio.load(f)
        
        # Resample just in case any files aren't 22050Hz
        if sr != 22050:
            waveform = torchaudio.functional.resample(waveform, sr, 22050)
        
        # 1. Convert to Mel
        mel = mel_transform(waveform)
        
        # 2. Convert to Log-Mel (Matcha-TTS clamps at 1e-5 to prevent negative infinity)
        log_mel = torch.log(torch.clamp(mel, min=1e-5))
        all_mels.append(log_mel)

    # Stitch all audio frames together into one massive tensor
    all_mels_tensor = torch.cat(all_mels, dim=-1)
    
    # Calculate the global statistics
    mel_mean = all_mels_tensor.mean().item()
    mel_std = all_mels_tensor.std().item()

    print("\n✅ Calculation Complete! Paste these into your aphasia.yaml:")
    print("-" * 40)
    print("data_statistics:")
    print(f"  mel_mean: {mel_mean:.6f}")
    print(f"  mel_std: {mel_std:.6f}")
    print("-" * 40)

In [24]:
WAV_DIR = "/rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected/preprocessed"
get_mel_stats(WAV_DIR)

🔍 Scanning directory: /rds/general/user/ak8224/home/emoji_project/data/Aphasia-Test/aphasia5/selected/preprocessed
📊 Found 77 files. Calculating Log-Mel Spectrograms...

✅ Calculation Complete! Paste these into your aphasia.yaml:
----------------------------------------
data_statistics:
  mel_mean: -1.355639
  mel_std: 1.636346
----------------------------------------
